In [2]:
# rescuing variants with high depth and the ones that were flagged only because of strandoddsratio

import pandas as pd

def rescue_variants(input_file, output_file):
    # Load the variants file
    df = pd.read_csv(input_file)
    
    # Define your "High Quality" thresholds for rescue
    DEPTH_THRESHOLD = 100 # Using a higher bar for SOR rescue
    AF_LOWER = 0.40       # Heterozygous range (~0.5)
    AF_UPPER = 0.60
    HOM_AF_THRESHOLD = 0.90 # Homozygous range (~1.0)

    def check_row(row):
        # We only care about rows currently marked 'low'
        if str(row['Confidence']).lower() != 'low':
            return row['Confidence']
        
        # 1. Identify the Filter Status
        # We accept 'PASS' OR 'StrandOddsRatio'
        current_filter = str(row['Filters 1']).upper()
        is_acceptable_filter = (current_filter == 'PASS' or current_filter == 'STRANDODDSRATIO')
        
        # 2. Check Depth 1 (The primary tube)
        # We want to be very sure, so we check if it meets a solid depth
        has_depth = row['Depth 1'] >= DEPTH_THRESHOLD
        
        # 3. Check Allele Fraction (Biological consistency)
        af = row['Allele fraction 1']
        is_het = (af >= AF_LOWER and af <= AF_UPPER)
        is_hom = (af >= HOM_AF_THRESHOLD)
        
        # 4. Check Replicate Status
        # Check if replicates 2-6 are empty/zero (the "lonely tube" scenario)
        other_reps_depth = sum([row.get(f'Depth {i}', 0) for i in range(2, 7) if pd.notnull(row.get(f'Depth {i}'))])

        # RESCUE LOGIC:
        # If (Filter is PASS or SOR) AND (High Depth) AND (50% or 100% AF)
        if is_acceptable_filter and has_depth and (is_het or is_hom):
            
            # Sub-label for clarity
            if current_filter == 'STRANDODDSRATIO':
                return 'high (rescued-SOR)'
            elif other_reps_depth == 0:
                return 'high (rescued-SingleRep)'
            else:
                return 'high (rescued)'
        
        return row['Confidence']

    # Apply the logic
    print(f"Processing {input_file}...")
    df['Confidence'] = df.apply(check_row, axis=1)
    
    # Summary of changes
    total_rescued = len(df[df['Confidence'].str.contains('rescued', na=False)])
    sor_rescued = len(df[df['Confidence'] == 'high (rescued-SOR)'])
    
    df.to_csv(output_file, index=False)
    print(f"Done! Rescued {total_rescued} variants total.")
    print(f"Specifically rescued {sor_rescued} variants that failed ONLY on StrandOddsRatio.")
    print(f"Results saved to: {output_file}")

# Usage
if __name__ == "__main__":
    rescue_variants('variants.csv', 'variants_final_rescued.csv')

Processing variants.csv...
Done! Rescued 297 variants total.
Specifically rescued 74 variants that failed ONLY on StrandOddsRatio.
Results saved to: variants_final_rescued.csv


In [3]:
# how many variants i have now
import pandas as pd

# Load your file
df = pd.read_csv('variants_final_rescued.csv')

# Count rows where Confidence is NOT 'low' (case-insensitive)
non_low_count = df[df['Confidence'].str.lower() != 'low'].shape[0]

print(f"Total non-low variants: {non_low_count}")

Total non-low variants: 367


In [6]:
# how many varints for each sample if i didn't rescue and if i do
import pandas as pd

def process_variants(input_file):
    # Load the variants file
    # If your file is Excel, use pd.read_excel(input_file)
    df = pd.read_csv(input_file)
    
    # 1. Filter out all 'low' confidence variants
    # Using str.lower() to ensure it catches 'Low', 'LOW', or 'low'
    filtered_df = df[df['Confidence'].str.lower() != 'low'].copy()
    
    # 2. Print the total count of variants kept
    print(f"Total non-low variants kept: {len(filtered_df)}")
    print("-" * 30)
    
    # 3. Print the number of rows for each unique sample
    # This uses the 'Sample' column to group and count
    sample_counts = filtered_df['Sample'].value_counts()
    
    print("Variant counts per sample (High/Medium only):")
    print(sample_counts.to_string())
    
    # 4. Optional: Save the cleaned table
    # filtered_df.to_csv('variants_filtered.csv', index=False)

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants.csv')

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants_final_rescued.csv')

Total non-low variants kept: 70
------------------------------
Variant counts per sample (High/Medium only):
Sample
control14            4
case1                3
control1             3
case8                3
case7                3
case3                3
case5                3
control3             3
control8             3
control5             3
control4             2
control2             2
PAH-29_S19_          2
case6                2
case10               2
PAH-21_S11_          2
control9             2
control7             2
case4                2
PAH-17_S7_           1
PAH-11_S1_           1
PAH-14_S4_           1
Contr-PAH-15_S30_    1
PAH-15_S5_           1
PAH-20_S10_          1
PAH-19_S9_           1
PAH-31_S21_          1
PAH-32_S22_          1
PAH-33_S23_          1
PAH-34_S24_          1
PAH-36_S26_          1
PAH-37_S27_          1
case2                1
PAH-26_S16_          1
control13            1
control10            1
case9                1
control12            1
control11 

In [19]:
# whats the difference between controls and samples if we use different filters
# i have 52 samples with 38 cases and 14 controls. none of the filters include all cases and no controls but at stingent filtering there are only cases (though not all) with only pathogenic variants 
import pandas as pd

def process_variants(input_file):
    # Load the variants file
    # If your file is Excel, use pd.read_excel(input_file)
    df = pd.read_csv(input_file)
    
    # 1. Filter out all 'low' confidence variants
    # Using str.lower() to ensure it catches 'Low', 'LOW', or 'low'
    filtered_df = df[
        #(df['Confidence'].str.lower() != 'low') & 
        #(df['Impact'].str.lower() != 'low') & 
        #(df['Consequence'].str.lower() != 'intron') &
        (df['Clinical significance'].str.lower() == 'pathogenic')
    ].copy() 
    
    # 2. Print the total count of variants kept
    print(f"Total non-low variants kept: {len(filtered_df)}")
    print("-" * 30)
    
    # 3. Print the number of rows for each unique sample
    # This uses the 'Sample' column to group and count
    sample_counts = filtered_df['Sample'].value_counts()
    
    print("Variant counts per sample (High/Medium only):")
    print(sample_counts.to_string())
    
    # 4. Optional: Save the cleaned table
    # filtered_df.to_csv('variants_filtered.csv', index=False)

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
    process_variants('variants.csv')

if __name__ == "__main__":
    # Change 'variants.csv' to your actual file name
#    process_variants('variants_final_rescued.csv')

Total non-low variants kept: 36
------------------------------
Variant counts per sample (High/Medium only):
Sample
PAH-35_S25_    3
PAH-31_S21_    3
PAH-22_S12_    2
PAH-13_S3_     2
PAH-12_S2_     2
PAH-36_S26_    2
PAH-37_S27_    2
PAH-33_S23_    2
PAH-25_S15_    2
PAH-30_S20_    2
PAH-32_S22_    2
PAH-26_S16_    2
PAH-27_S17_    2
PAH-18_S8_     1
PAH-16_S6_     1
PAH-14_S4_     1
PAH-19_S9_     1
PAH-29_S19_    1
PAH-28_S18_    1
case5          1
case6          1
Total non-low variants kept: 36
------------------------------
Variant counts per sample (High/Medium only):
Sample
PAH-35_S25_    3
PAH-31_S21_    3
PAH-22_S12_    2
PAH-13_S3_     2
PAH-12_S2_     2
PAH-36_S26_    2
PAH-37_S27_    2
PAH-33_S23_    2
PAH-25_S15_    2
PAH-30_S20_    2
PAH-32_S22_    2
PAH-26_S16_    2
PAH-27_S17_    2
PAH-18_S8_     1
PAH-16_S6_     1
PAH-14_S4_     1
PAH-19_S9_     1
PAH-29_S19_    1
PAH-28_S18_    1
case5          1
case6          1


In [38]:
# how many variants are called in controls under different filters

def filter_out_control_variants(output_file):
    
    # 1. Define how we uniquely identify a variant
    # Typically: Chromosome + Position + Ref + Alt
    variant_cols = ['Chromosome', 'Position', 'Ref', 'Alt']
    
    # 2. Identify Control Samples
    # We look for rows where the 'Sample' name contains 'Contr' (case-insensitive)
    is_control = df['Sample'].str.contains('Contr', case=False, na=False)
    control_df = df[is_control]
    
    # 3. Create a list of unique variants found in Controls
    # We drop duplicates to get a unique set of coordinates/alleles
    control_variants = control_df[variant_cols].drop_duplicates()
    
    # 4. Filter the main dataframe
    # We want to keep rows that are NOT in the control_variants set.
    # We use a 'left join' with an indicator to find non-matches.
    merged = df.merge(control_variants, on=variant_cols, how='left', indicator=True)
    
    # Rows where _merge is 'left_only' are variants NOT found in any control sample
    filtered_df = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    # 5. Save and Print Results
    filtered_df.to_csv(output_file, index=False)
    
    print(f"Original total rows: {len(df)}")
    print(f"Original unique variants: {df[variant_cols].drop_duplicates().shape[0]}")
    print(f"Unique variants found in Controls: {len(control_variants)}")
    print(f"Rows remaining after removing Control-linked variants: {len(filtered_df)}")
    print(f"Remaining unique variants: {filtered_df[variant_cols].drop_duplicates().shape[0]}")


# before any filtering
if __name__ == "__main__":
    print("\n before any filtering")
    import pandas as pd
    df = pd.read_csv('variants.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    #df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')

# after original filters
if __name__ == "__main__":
    print("\n after original filters")

    import pandas as pd
    df = pd.read_csv('variants.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')

# after rescuing
if __name__ == "__main__":
    print("\n after rescuing")
    import pandas as pd
    df = pd.read_csv('variants_final_rescued.csv')
    
    # select rows where Confidence is NOT 'low' (case-insensitive)
    df = df[df['Confidence'].str.lower() != 'low']
    filter_out_control_variants('variants_no_controls.csv')


 before any filtering
Original total rows: 545
Original unique variants: 45
Unique variants found in Controls: 23
Rows remaining after removing Control-linked variants: 71
Remaining unique variants: 22

 after original filters
Original total rows: 70
Original unique variants: 12
Unique variants found in Controls: 8
Rows remaining after removing Control-linked variants: 16
Remaining unique variants: 4

 after rescuing
Original total rows: 367
Original unique variants: 31
Unique variants found in Controls: 16
Rows remaining after removing Control-linked variants: 54
Remaining unique variants: 15


In [19]:
# saving all good variants after rescuing (includes multiple identifiers) for vep
import pandas as pd
df = pd.read_csv('variants_final_rescued.csv')
variant_cols = ['Chromosome', 'Position', 'Ref', 'Alt','Existing variation']
unique_variants = df[variant_cols].drop_duplicates()
df1 = unique_variants[['Existing variation']].copy()
df1['Existing variation'] = df1['Existing variation'].str.split(', ') # Step 1: Split into lists
df1 = df1.explode('Existing variation')                 # Step 2: Explode the lists
df1.dropna().to_csv("rsids.csv",index=False,header=False)
df1.dropna()

,Existing variation
0,rs772897
0,COSV108133564
2,rs1522306
2,COSV61015928
3,rs1042503
...,...
248,CM930536
260,rs118092776
260,CM981427
260,COSV61020094


In [99]:
# resulting vcf file - changing format from all in one columns to multiple 

# remove the first 10 rows with descriptions
import pandas as pd
df = pd.read_csv("vr_output.vcf",sep="\t",skiprows=11)
df.to_csv("vr_output1.vcf",sep='\t')

# method to expand columns
def expand_transcript_columns(input_file, output_file, columns_to_expand,sep1,sep2):
    """
    Expands multiple HGVS-style columns into a wide format.
    columns_to_expand: List of columns, e.g., ['HGVSc', 'HGVSp']
    """
    df = pd.read_csv(input_file,sep="\t")

    # This will hold all the newly created DataFrames for merging at the end
    expanded_dataframes = [df]

    for col in columns_to_expand:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found. Skipping.")
            continue

        print(f"Parsing {col}...")

        def parse_hgvs_string(hgvs_str):
            if pd.isna(hgvs_str) or hgvs_str == "":
                return {}
            
            entries = str(hgvs_str).split(sep2)
            row_data = {}
            for entry in entries:
                if sep1 in entry:
                    # transcript = 'NM_000277.3', change = 'c.1155C>G'
                    transcript, change = entry.split(sep1, 1)
                    # We prefix the column name to keep HGVSc separate from HGVSp
                    row_data[f"{col}|{transcript}"] = change
            return row_data

        # 1. Create a list of dictionaries for this specific column
        dicts = df[col].apply(parse_hgvs_string).tolist()
        
        # 2. Convert to a DataFrame
        new_df = pd.DataFrame(dicts)
        
        # 3. Store it for concatenation
        expanded_dataframes.append(new_df)

    # Combine the original DF with all new transcript-specific columns
    final_df = pd.concat(expanded_dataframes, axis=1)

    # drop original columns
    final_df = final_df.drop(columns=columns_to_expand)

    # Save results
    final_df.to_csv(output_file, index=False,sep="\t")
    
    new_col_count = len(final_df.columns) - len(df.columns)
    print(f"Done! Added {new_col_count} new transcript-specific columns.")

if __name__ == "__main__":
    # Specify the columns you want to burst open
    target_columns = ['INFO']
    
    expand_transcript_columns(
        'vr_output1.vcf', 
        'vr_output2.vcf', 
        target_columns,
        sep1='=',
        sep2=';'
    )
    
if __name__ == "__main__":
    # Specify the columns you want to burst open
    target_columns = ['INFO|HGVSg','INFO|HGVSc','INFO|HGVSp','INFO|SPDI','INFO|Variant_synonyms']
    
    expand_transcript_columns(
        'vr_output2.vcf', 
        'vr_output3.vcf', 
        target_columns,
        sep1=':',
        sep2=','
    )

Parsing INFO...
Done! Added 6 new transcript-specific columns.
Parsing INFO|HGVSg...
Parsing INFO|HGVSc...
Parsing INFO|HGVSp...
Parsing INFO|SPDI...
Parsing INFO|Variant_synonyms...
Done! Added 52 new transcript-specific columns.


In [117]:
# extracting only rsIDs from existing variation column
import pandas as pd
import re
df_variants = pd.read_csv("variants_color_coded.csv")
def extract_rsid(text):
    if pd.isna(text) or text == "":
        return ""
    match = re.search(r'\brs\d+\b', str(text))
    return match.group(0) if match else ""
source_col = 'Existing variation'
if source_col in df_variants.columns:
    df_variants['ID'] = df_variants[source_col].apply(extract_rsid)
else:
    print(f"Column {source_col} not found!")

pd.set_option('display.max_columns', None)

df_anno = pd.read_csv("vr_output3.vcf",sep="\t")
df_anno['#CHROM'] = df_anno['#CHROM'].replace(12, 'chr12')
df_anno = df_anno.rename(columns={'#CHROM': 'Chromosome', 'POS': 'Position','REF':'Ref','ALT':'Alt'})

merged_df1 = pd.merge(df_variants, df_anno, on=['ID'], how='left')
merged_df2 = pd.merge(df_variants, df_anno, on=['Chromosome','Position','Ref','Alt'], how='left')

#print(merged_df)
#merged_df.to_csv("with_anno.csv")
merged_df1

# need to fix the problem where i have multiple ref/alt alleles if i merge by rsID but if i merge by specific ref alt, my indels get messed up (some are not included)
# might want to still merge by positions but make exception for indels? or fix indels manually?

,Sample,Amplicon,Chromosome_x,Position_x,Ref_x,Alt_x,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,Chromosome_y,Position_y,Ref_y,Alt_y,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB
0,case1,pool2-5,chr12,102844478,T,G,case1,NaN,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,9 of 12,intron,modifier,COSV61021304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,TAATT,T/G,TATGT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,case1,pool2-9,chr12,102877572,G,A,case1,NaN,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,3 of 12,intron,modifier,rs2037639,benign,"25741868, 18937293, 23898865, 28706611",NaN,NaN,NaN,NaN,NaN,0.3165,0.2206,60,GCAAC,G/A,TCCTG,NaN,rs2037639,12.0,chr12,102877572.0,G,T,.,.,rs2037639,12-102877572-G-T,g.102877572G>T,c.338-22C>A,NaN,NaN,c.353-22C>A,NaN,NaN,c.353-22C>A,c.353-22C>A,c.353-22C>A,c.353-22C>A,c.245-22C>A,c.169-10909C>A,c.353-22C>A,c.353-22C>A,n.449-22C>A,c.353-22C>A,n.442-22C>A,c.337-22C>A,n.808-2307G>T,c.353-22C>A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102877571:G:T,RCV000254369,IVS3-22C>T,rs61037030,PAH_c.353-22C>T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,case1,pool2-9,chr12,102877572,G,A,case1,NaN,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,3 of 12,intron,modifier,rs2037639,benign,"25741868, 18937293, 23898865, 28706611",NaN,NaN,NaN,NaN,NaN,0.3165,0.2206,60,GCAAC,G/A,TCCTG,NaN,rs2037639,13.0,chr12,102877572.0,G,A,.,.,rs2037639,12-102877572-G-A,g.102877572G>A,c.338-22C>T,NaN,NaN,c.353-22C>T,NaN,NaN,c.353-22C>T,c.353-22C>T,c.353-22C>T,c.353-22C>T,c.245-22C>T,c.169-10909C>T,c.353-22C>T,c.353-22C>T,n.449-22C>T,c.353-22C>T,n.442-22C>T,c.337-22C>T,n.808-2307G>A,c.353-22C>T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102877571:G:A,RCV

In [126]:
# Create a comparison dataframe using an outer merge on all keys
comparison = pd.merge(
    df_variants, 
    df_anno, 
    on=['Chromosome', 'Position', 'Ref', 'Alt', 'ID'], 
    how='outer', 
    indicator=True
)

# Rows in df_variants (left) that didn't find a match in df_anno (right)
missing_in_anno = comparison[comparison['_merge'] == 'left_only']
print(missing_in_anno[missing_in_anno['Variant type'] =='SNV'].shape[0])
print(missing_in_anno[missing_in_anno['Variant type'] =='deletion'].shape[0])
print(missing_in_anno.shape)
missing_in_anno

39
5
(44, 100)


,Sample,Amplicon,Chromosome,Position,Ref,Alt,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID,Unnamed: 0,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB,_merge
0,control7,pool2-1,chr12,102838484,A,C,control7,NaN,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,13 of 13,NaN,3_prime_UTR,modifier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0,CTCAA,A/C,GTGTT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,control4,pool2-1,chr12,102838523,G,T,Contr-PAH-4_S29_,control4,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,13 of 13,NaN,3_prime_UTR,modifier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.0,AAAAT,G/T,ATACT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,control14,pool2-1,chr12,102838523,G,T,control14a,control14b,control14c,control14d,control14e,control14f,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,13 of 13,NaN,3_prime_UTR,modifier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.0,AAAAT,G/T,ATACT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
8,case3,pool1-4,chr12,102840329,G,T,case3,NaN,NaN,NaN,NaN,NaN,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,12 of 12,intron,modifier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.0,CCTAT,G/T,GCGAT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [129]:
import pandas as pd

# 1. Split df_variants into two groups
deletions = df_variants[df_variants['Variant type'].str.lower() == 'deletion']
others = df_variants[df_variants['Variant type'].str.lower() != 'deletion']

# 2. Perform the coordinate-based merge for non-deletions
# This is usually more precise for SNVs
merged_others = pd.merge(
    others, 
    df_anno, 
    on=['Chromosome', 'Position', 'Ref', 'Alt'], 
    how='left'
)

# 3. Perform the ID-based merge for deletions
# We drop ID from df_anno if it's already in the 'on' list to avoid duplicate columns
merged_deletions = pd.merge(
    deletions, 
    df_anno, 
    on=['ID'], 
    how='left'
)

# 4. Concatenate them back together
final_df = pd.concat([merged_others, merged_deletions], axis=0).reset_index(drop=True)

print(f"Merged {len(merged_others)} SNVs via coordinates and {len(merged_deletions)} deletions via ID.")

final_df.tail(10)
# i tried to mege snvs one way and deletions the other since deletions seem to be all the same but it made 1 more row than it was supposed to. im not sure why
# I might just add 5 deletions into excel 

Merged 356 SNVs via coordinates and 6 deletions via ID.


,Sample,Amplicon,Chromosome,Position,Ref,Alt,ID 1,ID 2,ID 3,ID 4,ID 5,ID 6,Variant type,Gene,Gene symbol,Feature,Feature type,Exon,Intron,Consequence,Impact,Existing variation,Clinical significance (ClinVar),PubMed,SIFT,PolyPhen,Codons,cDNA effect,Protein effect,1000 Genomes frequency,gnomAD frequency,Offset from primer end,5' context,Alleles,3' context,Indel length,ID_x,Unnamed: 0,ID_y,QUAL,FILTER,INFO|VARID,INFO|VCF,INFO|HGVSg|NC_000012.12,INFO|HGVSc|ENST00000307000.7,INFO|HGVSc|ENST00000549247.6,INFO|HGVSc|ENST00000551114.2,INFO|HGVSc|ENST00000553106.6,INFO|HGVSc|ENST00000635477.1,INFO|HGVSc|ENST00000635528.1,INFO|HGVSc|ENST00000906692.1,INFO|HGVSc|ENST00000906693.1,INFO|HGVSc|ENST00000906694.1,INFO|HGVSc|ENST00000906695.1,INFO|HGVSc|ENST00000906696.1,INFO|HGVSc|ENST00000906697.1,INFO|HGVSc|NM_000277.3,INFO|HGVSc|NM_001354304.2,INFO|HGVSc|ENST00000549111.5,INFO|HGVSc|XM_017019370.2,INFO|HGVSc|ENST00000551988.5,INFO|HGVSc|ENST00000550978.6,INFO|HGVSc|XR_007063428.1,INFO|HGVSc|ENST00000551337.5,INFO|HGVSc|ENST00000546844.1,INFO|HGVSc|ENST00000548677.2,INFO|HGVSc|ENST00000548928.1,INFO|HGVSc|ENST00000635500.1,INFO|HGVSc|ENST00000546708.5,INFO|HGVSc|ENST00000547319.1,INFO|HGVSp|ENSP00000303500.2,INFO|HGVSp|ENSP00000448059.1,INFO|HGVSp|ENSP00000489230.1,INFO|HGVSp|ENSP00000576751.1,INFO|HGVSp|ENSP00000576752.1,INFO|HGVSp|ENSP00000576753.1,INFO|HGVSp|ENSP00000576754.1,INFO|HGVSp|ENSP00000576755.1,INFO|HGVSp|ENSP00000576756.1,INFO|HGVSp|NP_000268.1,INFO|HGVSp|NP_001341233.1,INFO|HGVSp|XP_016874859.1,INFO|HGVSp|ENSP00000446658.1,INFO|HGVSp|ENSP00000489016.1,INFO|HGVSp|ENSP00000447620.1,INFO|SPDI|NC_000012.12,INFO|Variant_synonyms|ClinVar,INFO|Variant_synonyms|PAHdb,INFO|Variant_synonyms|ArchivedbSNP,INFO|Variant_synonyms|PhenCode,INFO|Variant_synonyms|dbSNPHGVS,INFO|Variant_synonyms|NM_001127179.2,INFO|Variant_synonyms|NM_000260.3,INFO|Variant_synonyms|NM_001127180.2,INFO|Variant_synonyms|NM_000260.4,INFO|Variant_synonyms|NM_001127180.1,INFO|Variant_synonyms|NM_001369365.1,INFO|Variant_synonyms|UniProt,INFO|Variant_synonyms|OMIM,INFO|Variant_synonyms|PharmGKB,Chromosome_x,Position_x,Ref_x,Alt_x,ID,Chromosome_y,Position_y,Ref_y,Alt_y
352,control14,pool2-9,chr12,102877572.0,G,A,control14a,control14b,control14c,control14d,control14e,control14f,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,NaN,3 of 12,intron,modifier,rs2037639,benign,"25741868, 18937293, 23898865, 28706611",NaN,NaN,NaN,NaN,NaN,0.3165,0.220600,60,GCAAC,G/A,TCCTG,NaN,rs2037639,13.0,rs2037639,.,.,rs2037639,12-102877572-G-A,g.102877572G>A,c.338-22C>T,NaN,NaN,c.353-22C>T,NaN,NaN,c.353-22C>T,c.353-22C>T,c.353-22C>T,c.353-22C>T,c.245-22C>T,c.169-10909C>T,c.353-22C>T,c.353-22C>T,n.449-22C>T,c.353-22C>T,n.442-22C>T,c.337-22C>T,n.808-2307G>A,c.353-22C>T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102877571:G:A,RCV000254369,IVS3-22C>T,rs61037030,PAH_c.353-22C>T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
353,control14,pool2-1,chr12,102838523.0,G,T,control14a,control14b,control14c,control14d,control14e,control14f,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,13 of 13,NaN,3_prime_UTR,modifier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58,AAAAT,G/T,ATACT,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
354,control14,pool2-7,chr12,102852922.0,C,T,control14a,control14b,control14c,control14d,control14e,control14f,SNV,ENSG00000171759,PAH,ENST00000553106,Transcript,7 of 13,NaN,synonymous,low,"rs1042503, CX056901, CX1618317, COSV61020278","benign, likely_pathogenic","25741868, 16251468, 23757202, 18937293, 218037...",NaN,NaN,gtG/gtA,NaN,NaN,0.3149,0.212000,110,CCAGC,C/T,ACAGG,NaN,rs1042503,6.0,rs1042503,.,.,"rs1042503,CX056901,CX1618317,COSV61020278",12-102852922-C-T,g.10